# S14A · ¿Cómo entrenar cuando la matriz ya no cabe en pandas?
**Universidad Central · Maestría en Analítica de Datos · Big Data 2026-2S · extensión opcional de S14, sin nota ni entrega obligatoria.**

**Caso:** una empresa de mantenimiento recibe ventanas de telemetría de equipos. A partir de 16 sensores en ocho instantes (128 características numéricas), estima si la siguiente ventana requiere inspección. Usaremos **datos sintéticos reproducibles**, no datos de equipos reales. La etiqueta se genera mediante una regla probabilística con ruido, no es una medición de fallas reales.

**Producto final:** tablas Bronze/Silver/Gold, indicadores, matriz de capacidad, modelos MLlib, predicciones sobre test, métricas, lista de prioridades y expediente JSON. Todo se calcula dentro de Databricks Free Edition; no se descarga un CSV masivo ni se reúne la tabla completa en Python.

**Mapa:** A0 configurar → A1 generar → A2 curar → A3 construir variables → A4 comparar memoria → A5 entrenar y seleccionar → A6 predecir y evaluar → A7 persistir y auditar → A8 reutilizar/limpiar.

**Dos recursos:** [Explorador S14A](https://jazaineam1.github.io/BigData2026/Presentaciones/s14a-databricks-escala.html) y este cuaderno. Los tiempos se miden en tu ejecución. La ruta opcional no modifica el taller final.

### Antes de comenzar
1. Entra a [Databricks Free Edition](https://www.databricks.com/learn/free-edition).
2. Descarga este IPYNB. Workspace → carpeta propia → Import → File; conecta Serverless. Environment → versión 4 o superior que admita pyspark.ml; no instales PySpark.
3. Catalog → identifica catálogo y esquema con CREATE TABLE. Ejecuta A0 y elige un tamaño. La primera vez usa 50.000; después 1.000.000 si tienes cuota. 5 y 25 millones requieren confirmación explícita y pueden exceder la cuota. No ejecutes una serie de tamaños automáticamente.
4. Ejecuta cada bloque en orden. Abre See performance → consulta → See query profile cuando esté disponible. No uses SparkContext, RDD, cache obligatorio ni toPandas completo.
5. Si Free Edition limita el cómputo, detén la ejecución; conserva el error. No contrates una cuenta ni uses una tarjeta por este ejercicio. [Límites oficiales](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations) · [MLlib Serverless](https://docs.databricks.com/aws/en/release-notes/serverless/environment-version/four).

<!-- PEDAGOGIA-RUTA-v1 -->
**Ruta esencial de aprendizaje · cuatro estaciones.** Lee las estaciones antes de ejecutar; amplía únicamente cuando hayas obtenido la evidencia.
<details><summary>Ver recorrido, comprobaciones y experimento formativo</summary>
<ol>
<li><strong>A0–A2:</strong> Elige 50.000 inicialmente y prepara Bronze/Silver y split. <em>Evidencia:</em> Conteos conciliados.</li>
<li><strong>A3–A4:</strong> Crea 128 variables y calcula capacidad; no uses toPandas completo. <em>Evidencia:</em> Bytes mínimos y muestra acotada.</li>
<li><strong>A5–A7:</strong> Entrena candidatos, selecciona en validación y evalúa test. <em>Evidencia:</em> Métricas, modelo y predicciones persistentes.</li>
<li><strong>Escalar:</strong> Solo si la cuota lo permite y con confirmación explícita. <em>Evidencia:</em> Evidencia de éxito o fallo auténtico, nunca estimación como corrida.</li>
</ol>
<p><strong>Experimento complementario:</strong> copia este código en una celda nueva cuando corresponda. No se ejecuta automáticamente y no sustituye el laboratorio original.</p>
<pre><code>filas=25000000
variables=128
train=.8
bytes_minimos=int(filas*train)*variables*8
presupuesto=16*1024**3
print(bytes_minimos/1024**3, &quot;GiB&quot;, &quot;excede:&quot;,bytes_minimos&gt;presupuesto)</code></pre>
<p><strong>Cómo se lee:</strong> El cálculo es un límite inferior de la matriz float64; no es el pico de memoria ni una prueba Spark.</p>
<p><strong>Modifica:</strong> Cambia filas o presupuesto; ejecuta A0–A7 en Databricks a una escala viable.</p>
<p><strong>Qué no permite concluir:</strong> No garantiza cuota Free Edition ni incapacidad universal de métodos locales incrementales.</p>
</details>
<!-- /PEDAGOGIA-RUTA-v1 -->

## A0 · Configuración y seguridad de escala
**Decisión:** empieza con 50.000 filas para validar permisos y MLlib. Una corrida de un millón es el objetivo normal de exploración, no una garantía de rendimiento. Las escalas grandes no se lanzan accidentalmente. El presupuesto de pandas se expresa en MiB y es una **restricción de la comparación**, no memoria detectada automáticamente.

- 50.000 filas: prueba de integración.
- 1 millón: ~0,76 GiB de matriz float64 train (80% aproximado).
- 5 millones: ~3,81 GiB.
- 25 millones: ~19,07 GiB, superior a un presupuesto de 16 GiB; puede ser inviable incluso en Free Edition.
  
El cálculo excluye etiquetas, índices, copias y memoria del algoritmo. pandas no entrena modelos por sí mismo; usamos scikit-learn como referencia local con una muestra. No afirmaremos que pandas nunca pueda procesar por bloques o que Spark garantice capacidad ilimitada.

In [ ]:
# A0 · Configura antes de ejecutar
import time, json, uuid, math
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
N = globals().get("LAB_N", 50000)
RAM_PANDAS_MIB = 512  # presupuesto comparativo; modifica a 8192 o 16384 si deseas
CONFIRMAR_GRANDE = False
SEGUNDO_MODELO = True  # árbol pequeño; pon False si la cuota no alcanza
SEMILLA = 42
PARTICIONES = 8
TEST_LOCAL = bool(globals().get("LAB_TEST_LOCAL", False))
assert N in (50000,1000000,5000000,25000000) or (TEST_LOCAL and N==5000)
assert N <= 1000000 or CONFIRMAR_GRANDE, "Confirma explícitamente una escala grande antes de continuar."
assert RAM_PANDAS_MIB > 0
assert "dbutils" in globals() or TEST_LOCAL, "Abre el notebook en Databricks; la prueba local es solo docente."
run_id = uuid.uuid4().hex[:10]
tiempos = {}
nombres = {}
if TEST_LOCAL:
    import tempfile
    carpeta_local = tempfile.mkdtemp(prefix="s14a_")
    formato = "parquet_local"
else:
    contexto = spark.sql("SELECT current_catalog() AS cat,current_schema() AS sch").first()
    CATALOGO = contexto["cat"]  # sustituye si no tienes CREATE TABLE
    ESQUEMA = contexto["sch"]
    formato = "delta"
def q(s):
    return "`"+s.replace("`","``")+"`"
def guardar(df, capa):
    inicio=time.perf_counter()
    if TEST_LOCAL:
        nombre=f"{carpeta_local}/{capa}"
        df.write.mode("overwrite").parquet(nombre)
        resultado=spark.read.parquet(nombre)
    else:
        nombre=".".join(q(x) for x in (CATALOGO,ESQUEMA,f"s14a_{run_id}_{capa}"))
        df.write.format("delta").mode("errorifexists").saveAsTable(nombre)
        resultado=spark.table(nombre)
    nombres[capa]=nombre
    tiempos["escribir_"+capa]=round(time.perf_counter()-inicio,3)
    print(capa, nombre, tiempos["escribir_"+capa], "s escritura")
    return resultado
def ver(df, n=10):
    if "dbutils" in globals() and not TEST_LOCAL:
        display(df.limit(n))
    else:
        df.show(n,truncate=False)
print("Spark",spark.version,"| escala",N,"| formato",formato,"| ejecución",run_id)
if not TEST_LOCAL: print("Catalog:",CATALOGO,"| Schema:",ESQUEMA)
print("Matriz prevista train GiB:",round(N*.8*128*8/1024**3,3))

**Cómo se lee:** el identificador distingue los activos de esta corrida; el formato Delta solo se anuncia en Databricks.

**Qué nos dice:** el cálculo no necesita paquetes adicionales ni fuentes externas.

**Qué NO permite concluir todavía:** que exista permiso de escritura hasta que A1 lo compruebe, ni que los 25 millones se puedan ejecutar.

**Qué error común:** configurar un clúster de pago o ejecutar la escala grande sin autorización.

## A1 · Ingesta: generar telemetría de forma distribuida y conservar Bronze
Una fila representa una ventana previa de 16 sensores. `spark.range(N)` crea IDs sin una lista Python. `xxhash64` y `pmod` generan valores reproducibles; son hashes, no simulaciones físicas de sensores. La etiqueta usa seis sensores actuales y ruido independiente. El ID solo sirve para identidad y reproducibilidad; no se entrega al modelo. Introducimos un valor nulo cada mil IDs y duplicamos nueve filas exactas para practicar calidad. Bronze conserva todo. La fuente es sintética, semilla 42.

**Qué tocar:** ejecuta, revisa el nombre de la tabla, el conteo y la muestra. Abre Catalog → esquema → tabla Bronze. La escritura es una acción real; no basta con construir un plan.

In [ ]:
# A1 · Ingesta Bronze: 16 sensores actuales y etiqueta futura simulada
def u(sal):
    return F.pmod(F.xxhash64(F.col("id"), F.lit(int(sal)), F.lit(SEMILLA)), F.lit(1000000)).cast("double")/1000000.0
SENSORES=[f"s{i:02d}" for i in range(16)]
base=spark.range(N,numPartitions=PARTICIONES).select("id",*[ (2*u(100+i)-1).alias(s) for i,s in enumerate(SENSORES)])
senal=1.5*F.col("s00")-1.3*F.col("s01")+1.1*F.col("s02")+0.9*F.col("s03")-0.8*F.col("s04")+0.6*F.col("s05")
base=base.withColumn("label",(senal+1.4*(2*u(7000)-1)>0).cast("double"))
raw=base.withColumn("s00",F.when(F.col("id")%1000==0,F.lit(None).cast("double")).otherwise(F.col("s00")))
raw=raw.unionByName(raw.filter(F.col("id").between(1,9)))
bronze=guardar(raw,"bronze")
n_bronze=bronze.count()
assert n_bronze==N+9
ver(bronze.select("id","s00","s01","s02","label").orderBy("id"))
print("Bronze:",n_bronze,"| esperadas:",N+9)

**Cómo se lee:** las primeras filas incluyen ID duplicado y lectura nula; el total debe ser N+9.

**Qué nos dice:** la ingesta se materializó y los datos crudos se conservaron.

**Qué NO permite concluir todavía:** la validez de las etiquetas para equipos reales: no hay fuente operacional.

**Qué error común:** construir un list de millones de diccionarios en el cliente o eliminar anomalías en Bronze.

## A2 · ETL: limpiar, separar rechazos y fijar train/validación/test
`dropDuplicates()` elimina filas idénticas; `isNotNull()` filtra sensores faltantes. La separación se obtiene de un hash distinto del que creó las etiquetas. Entrenamiento, validación y test son mutuamente excluyentes. No es una partición temporal; para equipos reales se necesitaría dividir por tiempo o por equipo según el uso.

**Esperado:** 9 duplicados, N/1000 nulos para nuestras escalas, N−N/1000 válidos. No se requiere que los porcentajes de split sean exactamente 80/10/10.

In [ ]:
# A2 · Silver: conciliación exacta y split estable
unicos=bronze.dropDuplicates()
rechazos=unicos.filter(F.col("s00").isNull())
validos=unicos.filter(F.col("s00").isNotNull())
bucket=F.pmod(F.xxhash64("id",F.lit(8801)),F.lit(10))
curados=validos.withColumn("split",F.when(bucket<8,"train").when(bucket==8,"valid").otherwise("test"))
silver=guardar(curados,"silver")
n_unicos=unicos.count()
n_rechazos=rechazos.count()
n_silver=silver.count()
assert n_bronze-n_unicos==9
assert n_rechazos==(N+999)//1000
assert n_silver==N-n_rechazos
assert n_silver+n_rechazos+9==n_bronze
assert silver.select("id").distinct().count()==n_silver
distribucion=silver.groupBy("split","label").count().orderBy("split","label")
grupos=[r.asDict() for r in distribucion.collect()]  # máximo seis filas
assert len(grupos)==6
print("Bronze",n_bronze,"| duplicados",9,"| rechazos",n_rechazos,"| Silver",n_silver)
ver(distribucion)
n_train=sum(int(x["count"]) for x in grupos if x["split"]=="train")
n_valid=sum(int(x["count"]) for x in grupos if x["split"]=="valid")
n_test=sum(int(x["count"]) for x in grupos if x["split"]=="test")
assert n_train+n_valid+n_test==n_silver

**Cómo se lee:** cada fila representa un grupo split-clase; los conteos suman Silver.

**Qué nos dice:** la separación y reglas de calidad son comprobables.

**Qué NO permite concluir todavía:** generalización futura, porque el split no está basado en tiempo ni equipo.

**Qué error común:** repartir nuevamente después de conocer el resultado del test.

## A3 · Gold: crear una matriz de 128 variables, sin pasar a pandas
Dieciséis sensores por ocho instantes = 128 características. El instante 0 usa la lectura actual; los otros siete son historias sintéticas correlacionadas con ella mediante perturbaciones deterministas. Son variables densas de punto flotante, no relleno de ceros ni replicación idéntica. Los historiales se generan en Spark a partir del ID y una semilla; en la realidad provendrían del almacenamiento histórico. No incluyen la etiqueta, el split ni el identificador como entrada del modelo.

Para limitar almacenamiento, Gold guarda **indicadores agregados**. Las 128 columnas se definen en un DataFrame a partir de Silver, y MLlib las consume durante fit. No materializamos otra copia masiva de 128 columnas en disco. `VectorAssembler` agrupa las columnas en un vector; la transformación se realiza en Spark.

**Qué observar:** muestra de cinco columnas, esquema de la matriz y consulta de agregación. Si la plataforma no ofrece Query Profile, usa explain y registra la limitación.

In [ ]:
# A3 · Variables en Spark y Gold de indicadores
FEATURES=[f"{s}_t{t}" for s in SENSORES for t in range(8)]
assert len(FEATURES)==128 and len(set(FEATURES))==128
exprs=[]
for i,s in enumerate(SENSORES):
    for t in range(8):
        x=F.col(s) if t==0 else 0.72*F.col(s)+0.28*(2*u(10000+i*8+t)-1)
        exprs.append(x.cast("double").alias(f"{s}_t{t}"))
matriz=silver.select("id","split","label",*exprs)
assert set(FEATURES).isdisjoint({"id","split","label"})
assert len(matriz.columns)==131
gold=guardar(silver.groupBy("split","label").agg(F.count("*").alias("ventanas"),F.avg("s01").alias("promedio_sensor_01")),"gold")
assert gold.agg(F.sum("ventanas")).first()[0]==n_silver
ver(gold.orderBy("split","label"))
ver(matriz.select("id","label",*FEATURES[:5]).orderBy("id"),5)
print("Variables:",len(FEATURES),"| filas de entrenamiento:",n_train)
matriz.select("id",*FEATURES[:3]).explain()

**Cómo se lee:** Gold resume seis grupos; la matriz conserva 128 columnas por fila.

**Qué nos dice:** los insumos de entrenamiento se construyen con expresiones nativas y no dependen de transferencias masivas.

**Qué NO permite concluir todavía:** que los historiales sintéticos representen sensores físicos o que el plan sea más rápido que otro.

**Qué error común:** usar id, label o split como características, o llamar a collect sobre todas las filas.

## A4 · Evidencia: ¿cabe en pandas?
El número de valores de la matriz de entrenamiento es filas × 128. Una matriz densa float64 requiere 8 bytes por valor; se calcula el mínimo exacto para esa representación. Mide en una muestra acotada de 1.000 filas el consumo de pandas y entrena un modelo local con 5.000 filas como control. La comparación de calidad con un modelo pequeño NO es una comparación de velocidad justa. El guardián de memoria bloquea explícitamente el traslado si supera el presupuesto; nunca provocamos OOM.

**Qué cambiar:** RAM_PANDAS_MIB en A0 define un presupuesto de la comparación, no impone una restricción al servidor Spark. En 25 millones la matriz train densa puede exceder 16 GiB. Si tu corrida tiene 50.000 filas, no hay demostración de exceder ese presupuesto; debes reconocerlo. float32, lectura por bloques, SGD incremental y algoritmos out-of-core son otras alternativas.

In [ ]:
# A4 · Prueba de memoria sin OOM y modelo de control acotado
import pandas as pd
from sklearn.linear_model import LogisticRegression as LRLocal
from sklearn.metrics import f1_score
train=matriz.filter(F.col("split")=="train")
valid=matriz.filter(F.col("split")=="valid")
test=matriz.filter(F.col("split")=="test")
muestra=train.select(*FEATURES).limit(1000).toPandas()
muestra_bytes=int(muestra.memory_usage(index=False,deep=True).sum())
bytes_minimos=int(n_train*len(FEATURES)*8)
presupuesto=int(RAM_PANDAS_MIB*1024**2)
excede=bytes_minimos>presupuesto
comparacion={"filas_train":n_train,"variables":128,"float64_bytes":bytes_minimos,
             "float64_GiB":round(bytes_minimos/1024**3,3),
             "muestra_filas":len(muestra),"muestra_bytes":muestra_bytes,
             "presupuesto_MiB":RAM_PANDAS_MIB,"excede":excede}
assert muestra_bytes==len(muestra)*128*8
print(json.dumps(comparacion,ensure_ascii=False,indent=2))
print("TRASLADO COMPLETO BLOQUEADO por presupuesto" if excede else "La matriz mínima cabe en el presupuesto; no demuestra incapacidad de pandas.")
# No hay toPandas masivo en ninguna rama. Muestra local acotada:
m_local=train.orderBy("id").select("label",*FEATURES).limit(5000).toPandas()
v_local=valid.orderBy("id").select("label",*FEATURES).limit(2000).toPandas()
local=LRLocal(max_iter=50)
local.fit(m_local[FEATURES],m_local["label"])
f1_local=float(f1_score(v_local["label"],local.predict(v_local[FEATURES]),average="macro"))
print("Control local: train",len(m_local),"valid",len(v_local),"F1 macro",round(f1_local,4))

**Cómo se lee:** float64_GiB es el mínimo de la matriz completa en esa representación; excede compara con el presupuesto explícito.

**Qué nos dice:** podemos justificar no materializar el dataset completo en pandas sin bloquear el entorno.

**Qué NO permite concluir todavía:** que pandas sea incapaz en cualquier máquina, con cualquier dtype, muestreo o método incremental; tampoco velocidad relativa.

**Qué error común:** confundir la extrapolación de RAM con un error de memoria observado o comparar la muestra local con un entrenamiento Spark completo como benchmark.

## A5 · Entrenar modelos Spark MLlib sobre todo train
**Pipeline:** VectorAssembler (128 columnas → vector features) y Estimator (aprende parámetros). Regresión logística optimiza una pérdida sobre observaciones repartidas; un árbol aprende reglas de división de variables. Usamos un árbol pequeño como segundo candidato, y ambos ven exactamente el mismo train. Si desactivas SEGUNDO_MODELO, solo entrenas LR. Seleccionamos usando F1 macro en validación, jamás test. El control mayoritario se calcula exclusivamente con train. No se modifica la configuración oculta de Serverless. `fit()` ejecuta el entrenamiento y `transform()` predice. Los tiempos medidos comprenden fit, no todo el pipeline. Ningún tiempo está prefijado.

In [ ]:
# A5 · Dos candidatos y selección por validación
def metrica(df):
    c={(int(r["label"]),int(r["prediction"])):int(r["count"]) for r in df.groupBy("label","prediction").count().collect()}
    m=[[c.get((i,j),0) for j in (0,1)] for i in (0,1)]
    total=sum(map(sum,m))
    f=[]
    for i in (0,1):
        vp=m[i][i]; fp=m[1-i][i]; fn=m[i][1-i]
        f.append(2*vp/(2*vp+fp+fn) if 2*vp+fp+fn else 0.0)
    return {"matriz":m,"filas":total,"accuracy":(m[0][0]+m[1][1])/total,"f1_macro":sum(f)/2}
def ensamblador():
    return VectorAssembler(inputCols=FEATURES,outputCol="features",handleInvalid="error")
candidatos={"logistica":LogisticRegression(featuresCol="features",labelCol="label",maxIter=12,regParam=0.02)}
if SEGUNDO_MODELO:
    candidatos["arbol"]=DecisionTreeClassifier(featuresCol="features",labelCol="label",maxDepth=4,maxBins=16,seed=SEMILLA)
modelos={}; validacion={}
for nombre,estimador in candidatos.items():
    print("Entrenando:",nombre,"| train:",n_train,"| 128 variables")
    inicio=time.perf_counter()
    modelo=Pipeline(stages=[ensamblador(),estimador]).fit(train.repartition(PARTICIONES))
    tiempos["fit_"+nombre]=round(time.perf_counter()-inicio,3)
    modelos[nombre]=modelo
    p=modelo.transform(valid).select("label","prediction")
    validacion[nombre]=metrica(p)
    assert validacion[nombre]["filas"]==n_valid
    print(nombre,"fit segundos:",tiempos["fit_"+nombre],"validación:",validacion[nombre])
mejor=max(validacion,key=lambda k:validacion[k]["f1_macro"])
mayoria=max((x for x in grupos if x["split"]=="train"),key=lambda x:x["count"])["label"]
print("Seleccionado SOLO con validación:",mejor,"| clase mayoritaria train:",mayoria)

**Cómo se lee:** el modelo se selecciona por F1 macro de validación; el tiempo es una medición de fit.

**Qué nos dice:** MLlib recibió todas las filas de train, no la muestra pandas; el modelo es común.

**Qué NO permite concluir todavía:** que haya un número determinado de máquinas, que el mejor modelo sea universal o que Spark sea más rápido.

**Qué error común:** elegir con el conjunto test, usar la muestra local como comparación de rendimiento o llamar entrenamiento distribuido a una animación.

## A6 · Predicción de test, matriz de confusión y priorización
El candidato elegido se aplica **una vez** al test. Persistimos predicciones en Delta, con probabilidad de inspección. La matriz de confusión usa filas reales y columnas predichas. El F1 macro da igual peso a las dos clases; ROC AUC mide discriminación según el ranking. La referencia mayoritaria siempre predice la etiqueta más frecuente en train. No ajustamos umbral con test.

**Acción:** consulta primero diez ejemplos, luego la matriz y el F1. Abre Query Profile debajo de la escritura o consulta si está disponible. Los registros de la tabla son datos sintéticos.

In [ ]:
# A6 · Inferencia de todo test, tabla Delta y métricas
from pyspark.ml.functions import vector_to_array
pred=modelos[mejor].transform(test).select("id","label","prediction",
    vector_to_array("probability")[1].alias("prob_inspeccion"))
predicciones=guardar(pred,"predicciones")
resultado=metrica(predicciones)
assert resultado["filas"]==n_test
assert predicciones.filter((F.col("prob_inspeccion")<0)|(F.col("prob_inspeccion")>1)|F.col("prob_inspeccion").isNull()).count()==0
baseline=metrica(test.select("label",F.lit(float(mayoria)).alias("prediction")))
auc=float(BinaryClassificationEvaluator(labelCol="label",rawPredictionCol="prob_inspeccion",metricName="areaUnderROC").evaluate(predicciones))
resultado["auc"]=auc
print("Test seleccionado:",mejor,resultado)
print("Baseline mayoría:",baseline)
ver(spark.createDataFrame([(i,j,resultado["matriz"][i][j]) for i in (0,1) for j in (0,1)],"real int,predicha int,registros long").orderBy("real","predicha"))
ver(predicciones.orderBy(F.desc("prob_inspeccion"),"id"),10)

**Cómo se lee:** predicciones tiene una fila por ID test; prob_inspeccion ordena el resultado.

**Qué nos dice:** la evaluación se hizo con etiquetas reservadas y la lista puede priorizar inspecciones simuladas.

**Qué NO permite concluir todavía:** que la probabilidad esté calibrada o que exista utilidad económica real; faltan costos, calibración y datos reales.

**Qué error común:** usar F1 de validación como test, confundir probabilidad con certeza o aplicar umbrales sin criterio operacional.

## A7 · Producto persistente: métricas y expediente
Guardamos una tabla de métricas y un expediente estructurado con conteos, tiempos medidos, versiones de entorno, tablas, presupuesto y decisión. Guardar predicciones no equivale a persistir el modelo. Si quieres reutilizar el artefacto del modelo, A8 permite registro opcional con MLflow; primero valida que tu entorno lo admite. No se solicita servicio de inferencia ni gasto adicional.

**SQL:** copia el nombre completo de Gold y predicciones, abre Catalog → Sample Data y usa una celda SQL. `SELECT * FROM nombre_completo LIMIT 10`. Puedes consultar mediante spark.sql si no hay SQL Warehouse disponible.

In [ ]:
# A7 · Auditoría y evidencia consultable
metricas=spark.createDataFrame([(mejor,"test",float(resultado["f1_macro"]),float(resultado["accuracy"]),float(auc)),
    ("mayoria","test",float(baseline["f1_macro"]),float(baseline["accuracy"]),None)],
    "modelo string, conjunto string, f1_macro double, accuracy double, auc double")
tabla_metricas=guardar(metricas,"metricas")
ver(tabla_metricas)
expediente={"sesion":"S14A","fuente":"telemetria sintetica reproducible","semilla":SEMILLA,
    "spark":spark.version,"formato":formato,"run_id":run_id,"N":N,"bronze":n_bronze,"rechazos":n_rechazos,
    "duplicados":9,"silver":n_silver,"train":n_train,"valid":n_valid,"test":n_test,
    "memoria":comparacion,"control_local":{"train":len(m_local),"valid":len(v_local),"f1_macro_valid":f1_local},
    "validacion":validacion,"seleccionado":mejor,"test_resultado":resultado,"baseline":baseline,
    "tiempos_segundos":tiempos,"activos":dict(nombres),
    "limite":"Fuente sintética; presupuesto de RAM declarado; no implica hardware real ni garantía de cuota."}
print(json.dumps(expediente,ensure_ascii=False,indent=2))
print("SQL:",f"SELECT * FROM {nombres['gold']} LIMIT 10" if not TEST_LOCAL else "Parquet local")
print("SQL:",f"SELECT * FROM {nombres['predicciones']} ORDER BY prob_inspeccion DESC LIMIT 10" if not TEST_LOCAL else "Parquet local")

**Cómo se lee:** los nombres de las tablas y tiempos provienen de esta corrida.

**Qué nos dice:** se puede reproducir y auditar la ejecución, y consultar productos sin dejar el notebook abierto.

**Qué NO permite concluir todavía:** que el modelo esté registrado o que las tablas Parquet locales prueben Delta.

**Qué error común:** presentar un expediente fabricado, publicar datos o credenciales, o dejar cómputo activo.

## A8 · Opcional: reutilizar el modelo y limpiar tus tablas
Si el Workspace admite MLflow, selecciona `REGISTRAR_MLFLOW=True` para guardar el PipelineModel entrenado como artefacto de un run. No registra automáticamente un endpoint ni exige un Model Registry. Si falla, conserva el error: la prueba principal de fit y las predicciones no dependen de esto. **No ejecutes la limpieza hasta guardar evidencia**. El código solo borra tablas cuyos nombres generó esta ejecución. En una corrida nueva, conserva o elimina las tablas anteriores manualmente en Catalog. No se borran activos de otras personas.

**Reto exploratorio:** modifica un solo tamaño de A0; conserva las 128 variables, semilla y modelos. Compara dos expedientes propios: número de filas, GiB calculados, tiempos observados, cuotas, F1. No es una entrega obligatoria. Si deseas medir velocidad, debes repetir bajo condiciones equivalentes y distinguir el arranque del cómputo. No hay benchmark automatizado.

In [ ]:
# A8 · Opcional, no ejecutar por defecto operaciones de registro o borrado
REGISTRAR_MLFLOW=False
LIMPIAR=False
if REGISTRAR_MLFLOW and not TEST_LOCAL:
    import mlflow
    import mlflow.spark
    with mlflow.start_run(run_name="S14A_"+run_id) as run:
        mlflow.log_params({"N":N,"features":128,"modelo":mejor})
        mlflow.log_metric("f1_macro_test",float(resultado["f1_macro"]))
        info=mlflow.spark.log_model(modelos[mejor],artifact_path="pipeline")
        print("Modelo registrado como artefacto del run:",run.info.run_id,info.model_uri)
if LIMPIAR:
    if TEST_LOCAL:
        import shutil
        shutil.rmtree(carpeta_local)
    else:
        for nombre in nombres.values():
            assert ("s14a_"+run_id+"_") in nombre
            spark.sql("DROP TABLE IF EXISTS "+nombre)
    print("Activos de la corrida eliminados.")
else:
    print("Activos conservados. Detén el cómputo si no vas a continuar.")

## Cierre
- Spark ejecuta cálculos sobre DataFrames sin materializar toda la matriz en el cliente.
- Un clasificador MLlib puede ajustar un modelo común usando datos particionados.
- Delta permite mantener los activos de un flujo reproducible; SQL consulta resultados.
- Free Edition no garantiza poder entrenar el tamaño extremo. Si la cuota se agota, conserva la evidencia real y no reportes una corrida inexistente.
- La prueba docente local no es prueba de Databricks, ni demuestra un clúster de varias máquinas.

[Documentación de MLlib](https://docs.databricks.com/aws/en/machine-learning/train-model/mllib) · [Spark ML Pipelines](https://spark.apache.org/docs/4.0.1/ml-pipeline.html) · [Databricks Free](https://docs.databricks.com/aws/en/getting-started/free-edition-limitations).

## Laboratorio ampliado · Preflight de entrenamiento y evidencia de capacidad

### No confundir tres pruebas
Una estimación dimensional de N×D×bytes es un límite inferior de almacenamiento denso, no un pico de memoria de pandas ni garantía de capacidad Spark. Una ejecución pequeña permite comprobar compatibilidad de API, tipos y permisos; no demuestra escalamiento. La prueba de capacidad del laboratorio A0–A7 requiere un fit terminado sobre N efectivo de train, dimensiones reales, entorno, tiempo, métricas y persistencia, además de un presupuesto local explícito.

La extensión calcula tamaños teóricos y ejecuta, solo si se activa con una sesión Spark válida, un fit mínimo de regresión logística sobre datos sintéticos generados con range. El split por ID evita superposición; las etiquetas dependen de dos variables y una condición determinista de ruido. Se ajusta solo con train y se evalúa una vez con test. No se llama a pandas ni collect para entrenar. Un error de API en serverless se reporta como bloqueo de compatibilidad; no se cambia a una ruta local silenciosamente para certificar Databricks.

### Cómo interpretar
La salida pequeña registra filas de entrenamiento, dimensión, tiempo y accuracy. No implica que el modelo generalice a SECOP. Antes de ampliar N, comprueba cuota y autorización y usa la ruta principal A0–A7. No es necesario provocar un OOM intencional.

**Reto:** define un presupuesto local explícito, calcula el umbral teórico de N para D=128 float64, y especifica los registros que demostrarían que el fit grande terminó realmente.


In [ ]:
import time
for n in (1000,1000000,10000000,20000000):
    print(n,"filas, límite inferior GiB",n*128*8/2**30)
assert abs(20000000*128*8/2**30-19.073486328125)<1e-9
EJECUTAR_PREFLIGHT_ML_S14A=False
if EJECUTAR_PREFLIGHT_ML_S14A:
    from pyspark.sql import functions as F
    from pyspark.ml.feature import VectorAssembler
    from pyspark.ml.classification import LogisticRegression
    from pyspark.ml.evaluation import MulticlassClassificationEvaluator
    if "spark" not in globals(): raise RuntimeError("Abre el compute de práctica")
    muestra=(spark.range(1000)
             .withColumn("x1",(F.col("id")%11).cast("double"))
             .withColumn("x2",(F.col("id")%7).cast("double"))
             .withColumn("label",((F.col("x1")+F.col("x2")+F.col("id")%3)>9).cast("double")))
    vector=VectorAssembler(inputCols=["x1","x2"],outputCol="features").transform(muestra)
    train=vector.filter(F.col("id")%5!=0)
    test=vector.filter(F.col("id")%5==0)
    ntrain,ntest=train.count(),test.count()
    assert (ntrain,ntest)==(800,200)
    assert train.select("id").join(test.select("id"),"id").count()==0
    inicio=time.perf_counter()
    modelo=LogisticRegression(maxIter=5,regParam=.01).fit(train)
    segundos=time.perf_counter()-inicio
    pred=modelo.transform(test)
    accuracy=MulticlassClassificationEvaluator(metricName="accuracy").evaluate(pred)
    print({"version_spark":spark.version,"train":ntrain,"test":ntest,
           "dimension":2,"fit_segundos":segundos,"accuracy":accuracy,
           "estado":"preflight pequeño, NO evidencia de escala"})
else:
    print("Fit de preflight pendiente; solo cálculo dimensional ejecutado.")
